# Training for Infill Model
Imports, params, loading dataset

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("/notebooks/"))  # make src/ importable

import torch
from pathlib import Path
import json
from ml.datasets import download_sources, load_paths, load_records, dataset_status
from ml.preprocess import build_token_cache, DEFAULT_CACHE_DIR
from ml.model import build_model, load_model, append_epoch_to_config
from ml.generation_trainer import train_generation
from ml.tokenizer import (
    VOCAB_SIZE
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
print(f"Vocabulary size: {VOCAB_SIZE} (includes infill + chord-symbol tokens)")

In [ ]:
# Training Configuration
EPOCHS = 2000
BATCH_SIZE = 8
LR = 1e-3
# FIM_RATIO = 0.5  # 50% FIM examples, 50% standard examples

# --- Dataset sources --------------------------------------------------------
# Add "pdmx" / "wikifonia" once their archives are downloaded/extracted (below).
SOURCES = ["openscore"]

# OpenScore: how many scores to pull from the GitHub repo (it has ~1462).
OPENSCORE_MAX = 10000

# PDMX (~250K, Zenodo record 14648209): extract the archive once, point at it here.
# A capped, high-quality subset is selected (rated + deduplicated, top by rating).
PDMX_ROOT = None       # e.g. "/storage/data/pdmx/PDMX"
PDMX_MAX = 20000

# Wikifonia / EWLD lead sheets: a directory of .mxl/.xml files.
WIKIFONIA_ROOT = None  # e.g. "/storage/data/wikifonia"

# Token-cache build: music21 parse processes (Linux fork -> safe to raise).
CACHE_WORKERS = 4

In [ ]:
# Download / locate each selected source (per-source kwargs), then merge paths.
_noprog = lambda i, n: None
for src in SOURCES:
    if src == "openscore":
        download_sources(["openscore"], log=print, progress=_noprog, max_files=OPENSCORE_MAX)
    elif src == "pdmx":
        download_sources(["pdmx"], log=print, progress=_noprog, max_files=PDMX_MAX, pdmx_root=PDMX_ROOT)
    elif src == "wikifonia":
        download_sources(["wikifonia"], log=print, progress=_noprog, wikifonia_root=WIKIFONIA_ROOT)

print(dataset_status(SOURCES))
paths = load_paths(SOURCES)
print(f"Using {len(paths)} score files across {SOURCES}.")

In [ ]:
# Parse every score ONCE into the on-disk token cache (incremental + parallel).
# Training then streams tokenized shards — no music21 parsing in the epoch loop.
# Building from records keeps source/metadata provenance; re-runs only parse new files.
store = build_token_cache(load_records(SOURCES), n_workers=CACHE_WORKERS)
print(f"Token cache ready: {len(store)} scores at {DEFAULT_CACHE_DIR}")

<hr>

### Build model

In [4]:
# Build infill model with expanded vocabulary
MODEL_NAME = 'GEN_D512_H16_L8'

print(f"Building {MODEL_NAME} model...")
model, cfg = build_model(name=MODEL_NAME, override_cfg=True)#load_model(device=DEVICE)
print(f"Infill model parameters: {model.num_params():,}")
print(f"Vocab size: {model.cfg['vocab_size']}")

model_log = []
def log_fn(msg):
    print(msg)
    model_log.append(msg)
        
def on_epoch(epoch, train_loss, val_loss):
    print(f"  → epoch {epoch}  train={train_loss:.4f}  val={val_loss:.4f}")
    append_epoch_to_config(MODEL_NAME, cfg, train_loss, val_loss)

Building GEN_D512_H16_L8 model...
Infill model parameters: 20,425,728
Vocab size: 463


<hr>

### Train model

In [ ]:
stop_flag = [False]  # set stop_flag[0] = True to interrupt training
train_generation(
    name=MODEL_NAME,
    model=model,
    data_paths=paths,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    device=DEVICE,
    log=log_fn,
    on_epoch=on_epoch,
    stop_flag=stop_flag,
    # fim_ratio=FIM_RATIO,
    resume_optimizer=False,
    cache_workers=CACHE_WORKERS,
)